# DL4NLP Assignment 2 — Colab Training with Hugging Face Trainer

This notebook trains the completed `A2Transformer` for **Task 2.1** using Hugging Face `Trainer`.

The Assignment 2 instructions explicitly allow using a Hugging Face Trainer. The Transformer itself remains your implementation; `Trainer` only handles the generic training/evaluation/checkpoint loop.

## Expected Google Drive layout

Place these files under:

`MyDrive/DL4NLP/A2/`

```text
A2/
├── A1_skeleton.py
├── A2_skeleton.py
├── train.txt
├── val.txt
└── trainer_output_50k/
    └── tokenizer.pkl
```

- `A1_skeleton.py`: your completed A1 code, needed to unpickle/load `A1Tokenizer`
- `A2_skeleton.py`: your completed A2 Tasks 1.1–1.5
- `train.txt`, `val.txt`: the same data as A1
- `tokenizer.pkl`: the 50k A1 tokenizer

The notebook copies code/data to `/content` for faster access. Hugging Face checkpoints and the final model are saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Locate files and copy code/data to local Colab storage

In [ ]:
from pathlib import Path
import shutil
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

PROJECT_DIR = Path("/content/drive/MyDrive/DL4NLP/A2")

A1_CODE = PROJECT_DIR / "A1_skeleton.py"
A2_CODE = PROJECT_DIR / "A2_skeleton.py"
TRAIN_FILE_DRIVE = PROJECT_DIR / "train.txt"
VAL_FILE_DRIVE = PROJECT_DIR / "val.txt"
TOKENIZER_FILE = PROJECT_DIR / "trainer_output_50k" / "tokenizer.pkl"

OUTPUT_DIR = PROJECT_DIR / "trainer_output_a2_50k"
FINAL_DIR = OUTPUT_DIR / "final_model"

required_files = [
    A1_CODE,
    A2_CODE,
    TRAIN_FILE_DRIVE,
    VAL_FILE_DRIVE,
    TOKENIZER_FILE,
]

for path in required_files:
    assert path.exists(), f"Missing required file: {path}"

shutil.copy2(A1_CODE, "/content/A1_skeleton.py")
shutil.copy2(A2_CODE, "/content/A2_skeleton.py")
shutil.copy2(TRAIN_FILE_DRIVE, "/content/train.txt")
shutil.copy2(VAL_FILE_DRIVE, "/content/val.txt")

print("Project directory:", PROJECT_DIR)
print("Trainer output:", OUTPUT_DIR)

## 2. Imports and GPU check

Colab normally already contains `transformers`, `datasets`, and `accelerate`.

If imports fail in your runtime, uncomment and run the following line once, then restart the runtime:

```python
%pip install -q transformers datasets accelerate
```

In [ ]:
import sys
sys.path.insert(0, "/content")

import json
import math
import torch
import transformers
import datasets

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import Trainer, TrainingArguments

from A1_skeleton import A1Tokenizer
from A2_skeleton import A2ModelConfig, A2Transformer

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Prefer bf16 where the GPU really supports it; otherwise use fp16 on CUDA.
USE_BF16 = bool(
    torch.cuda.is_available()
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)

print("bf16:", USE_BF16)
print("fp16:", USE_FP16)

## 3. Hyperparameters

These are experiment choices, not fixed assignment requirements.

The assignment recommends a small Transformer, for example a couple of layers.

In [ ]:
# Model
HIDDEN_SIZE = 256
INTERMEDIATE_SIZE = 1024
NUM_ATTENTION_HEADS = 8
NUM_HIDDEN_LAYERS = 2
ROPE_THETA = 10_000.0
RMS_NORM_EPS = 1e-6

# Training
LEARNING_RATE = 3e-4
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 50

# Set these to small numbers only for a quick integration test.
# Leave both as None for the final full-data training run.
DEBUG_TRAIN_EXAMPLES = None
DEBUG_VAL_EXAMPLES = None

# Resume support:
# - None: start a new run
# - True: Trainer resumes from the latest checkpoint in OUTPUT_DIR
# - or a string/path to a particular checkpoint directory
RESUME_FROM_CHECKPOINT = None

print("Head dimension:", HIDDEN_SIZE // NUM_ATTENTION_HEADS)
assert HIDDEN_SIZE % NUM_ATTENTION_HEADS == 0

## 4. Load the A1 tokenizer and the A1 dataset

Using the same 50k tokenizer makes the A1 RNN and A2 Transformer directly comparable at the tokenization/vocabulary level.

In [ ]:
tokenizer = A1Tokenizer.from_file(str(TOKENIZER_FILE))

print("Vocabulary size:", len(tokenizer))
print("Tokenizer max length:", tokenizer.model_max_length)

dataset = load_dataset(
    "text",
    data_files={
        "train": "/content/train.txt",
        "val": "/content/val.txt",
    },
)

dataset = dataset.filter(lambda x: x["text"].strip() != "")

if DEBUG_TRAIN_EXAMPLES is not None:
    dataset["train"] = dataset["train"].select(
        range(min(DEBUG_TRAIN_EXAMPLES, len(dataset["train"])))
    )

if DEBUG_VAL_EXAMPLES is not None:
    dataset["val"] = dataset["val"].select(
        range(min(DEBUG_VAL_EXAMPLES, len(dataset["val"])))
    )

print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["val"]))
print("Example:", dataset["train"][0]["text"][:200])

## 5. Create the A2 Transformer

This is the model you implemented in Tasks 1.1–1.5. `Trainer` does not replace any Transformer component.

In [ ]:
config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    rope_theta=ROPE_THETA,
    hidden_act="silu",
    max_position_embeddings=tokenizer.model_max_length,
    rms_norm_eps=RMS_NORM_EPS,
)

model = A2Transformer(config)

num_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters:     {num_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print("Vocabulary size:", model.config.vocab_size)

## 6. Custom data collator

The dataset currently contains raw `text`, but `A2Transformer.forward()` expects `input_ids` and optional `labels`.

The collator:
1. tokenizes each batch,
2. pads/truncates using the A1 tokenizer,
3. copies `input_ids` into `labels`,
4. changes padding labels to `-100` so they do not contribute to cross-entropy.

`remove_unused_columns=False` will be set in `TrainingArguments`, because otherwise Trainer would remove the raw `text` column before this collator sees it.

In [ ]:
class A2DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, examples):
        texts = [example["text"] for example in examples]

        encoding = self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"]

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "labels": labels,
        }


data_collator = A2DataCollator(tokenizer)

## 7. Real-data integration test

Run this before the long training job. It verifies:

raw text → A1 tokenizer → collator → A2 Transformer → finite loss → backward.

In [ ]:
test_batch = data_collator([
    dataset["train"][0],
    dataset["train"][1],
])

model = model.to(device)
model.train()
model.zero_grad(set_to_none=True)

test_input_ids = test_batch["input_ids"].to(device)
test_labels = test_batch["labels"].to(device)

test_output = model(
    input_ids=test_input_ids,
    labels=test_labels,
)

print("input_ids:", test_input_ids.shape)
print("logits:", test_output.logits.shape)
print("loss:", float(test_output.loss))

assert test_output.logits.shape[:2] == test_input_ids.shape
assert test_output.logits.shape[-1] == len(tokenizer)
assert torch.isfinite(test_output.loss)

test_output.loss.backward()
model.zero_grad(set_to_none=True)

print("Real-data integration test passed.")

## 8. Hugging Face TrainingArguments

Important settings:

- `eval_strategy="epoch"`: validate after every epoch
- `save_strategy="epoch"`: save a checkpoint after every epoch
- `load_best_model_at_end=True`: restore the checkpoint with the lowest validation loss
- `prediction_loss_only=True`: avoids storing enormous `(B, N, V)` logits during validation
- `remove_unused_columns=False`: preserves raw `text` for the custom collator
- `fp16`/`bf16`: selected automatically based on the GPU

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    optim="adamw_torch",

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=3,

    fp16=USE_FP16,
    bf16=USE_BF16,

    # We only need eval_loss for Task 2.1/perplexity.
    # Do not accumulate 50k-vocabulary logits during validation.
    prediction_loss_only=True,

    # The dataset contains raw "text", which our collator needs.
    remove_unused_columns=False,

    # Keep this simple/robust on Colab.
    dataloader_num_workers=0,

    report_to="none",
    seed=42,
)

print(training_args)

## 9. Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=data_collator,
)

print("Trainer ready.")

## 10. Train

For a fresh run:

```python
RESUME_FROM_CHECKPOINT = None
```

If Colab disconnects and checkpoints already exist in `OUTPUT_DIR`, set:

```python
RESUME_FROM_CHECKPOINT = True
```

and rerun this cell. Hugging Face Trainer will resume from the latest checkpoint.

In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)

print("Training complete.")
print(train_result)

## 11. Trainer validation loss and perplexity

For a causal language model trained with natural-log cross-entropy:

`perplexity = exp(cross_entropy)`

Because `load_best_model_at_end=True`, this evaluates the best validation-loss checkpoint selected during training.

In [ ]:
eval_result = trainer.evaluate()

trainer_val_loss = eval_result["eval_loss"]

try:
    trainer_val_ppl = math.exp(trainer_val_loss)
except OverflowError:
    trainer_val_ppl = float("inf")

print(f"Trainer validation cross-entropy: {trainer_val_loss:.4f}")
print(f"Trainer validation perplexity:    {trainer_val_ppl:.2f}")
print(eval_result)

## 12. Save the final/best model

`Trainer` already saves epoch checkpoints. This additionally saves the best model loaded at the end into a stable `final_model/` directory together with the A1 tokenizer and experiment metadata.

In [ ]:
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR))
tokenizer.save(str(FINAL_DIR / "tokenizer.pkl"))

experiment_info = {
    "vocab_size": len(tokenizer),
    "hidden_size": HIDDEN_SIZE,
    "intermediate_size": INTERMEDIATE_SIZE,
    "num_attention_heads": NUM_ATTENTION_HEADS,
    "num_hidden_layers": NUM_HIDDEN_LAYERS,
    "rope_theta": ROPE_THETA,
    "rms_norm_eps": RMS_NORM_EPS,
    "model_max_length": tokenizer.model_max_length,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    "num_parameters": num_parameters,
    "trainer_validation_cross_entropy": trainer_val_loss,
    "trainer_validation_perplexity": trainer_val_ppl,
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

with open(FINAL_DIR / "experiment_info.json", "w") as f:
    json.dump(experiment_info, f, indent=2)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Final model saved to:", FINAL_DIR)

## 13. Reload verification

This verifies that the saved model and tokenizer can be loaded independently from Drive.

In [ ]:
reloaded_tokenizer = A1Tokenizer.from_file(
    str(FINAL_DIR / "tokenizer.pkl")
)

reloaded_model = A2Transformer.from_pretrained(
    str(FINAL_DIR)
).to(device)

reloaded_model.eval()

reload_batch = data_collator([
    {"text": "She lives in San"}
])

reload_ids = reload_batch["input_ids"].to(device)

with torch.no_grad():
    reload_output = reloaded_model(reload_ids)

print("Reloaded logits shape:", reload_output.logits.shape)

assert reload_output.logits.shape[-1] == len(reloaded_tokenizer)
assert torch.isfinite(reload_output.logits).all()

print("Reload verification passed.")

## 14. Optional: exact token-weighted validation perplexity

`Trainer.evaluate()` is sufficient for the assignment. This extra evaluator is useful if you want the validation cross-entropy weighted exactly by the number of valid next-token targets, matching the token-level interpretation of perplexity.

It does **not** train the model; it only evaluates the final saved model.

In [ ]:
@torch.no_grad()
def token_weighted_perplexity(model, eval_dataset, batch_size):
    loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=data_collator,
    )

    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        output = model(
            input_ids=input_ids,
            labels=labels,
        )

        # A2Transformer predicts labels[:, 1:].
        valid_targets = (labels[:, 1:] != -100).sum().item()

        total_nll += output.loss.item() * valid_targets
        total_tokens += valid_targets

    cross_entropy = total_nll / total_tokens
    perplexity = math.exp(cross_entropy)

    return cross_entropy, perplexity


exact_val_loss, exact_val_ppl = token_weighted_perplexity(
    reloaded_model,
    dataset["val"],
    EVAL_BATCH_SIZE,
)

print(f"Token-weighted validation cross-entropy: {exact_val_loss:.4f}")
print(f"Token-weighted validation perplexity:    {exact_val_ppl:.2f}")

with open(FINAL_DIR / "final_metrics.json", "w") as f:
    json.dump(
        {
            "trainer_validation_cross_entropy": trainer_val_loss,
            "trainer_validation_perplexity": trainer_val_ppl,
            "token_weighted_validation_cross_entropy": exact_val_loss,
            "token_weighted_validation_perplexity": exact_val_ppl,
        },
        f,
        indent=2,
    )

print("Saved:", FINAL_DIR / "final_metrics.json")

## If CUDA runs out of memory

First reduce only the batch sizes:

```python
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
```

If needed, reduce them to `2`.

Do not change the architecture unless you intentionally want to run a different model experiment.

If mixed precision produces non-finite loss, disable the selected mixed-precision mode and restart the runtime.